In [1]:
from google.colab import files
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import classification_report

In [2]:
upload = files.upload()

Saving battery_synthetic_dataset.csv to battery_synthetic_dataset.csv


In [3]:
df = pd.read_csv("battery_synthetic_dataset.csv")

In [4]:
data = df.copy()

x = data.drop("label", axis=1)
y = data["label"]

In [5]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [6]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [7]:
input_size = x_train.shape[1]
hidden1_size = 5
hidden2_size = 7
hidden3_size = 10
hidden4_size = 5
output_size = 1

In [8]:
def sigmoid(x):
    return 1/(1+np.exp(-x))
def sigmoid_derivative(x):
    return x*(1-x)
def relu(x):
    return np.maximum(0, x)
def relu_derivative(x):
    return (x > 0).astype(float)

In [9]:
rng = np.random.default_rng(42)

def he_init(fan_in, fan_out):
    return rng.standard_normal((fan_in, fan_out)) * np.sqrt(2/fan_in)

In [10]:
weight_input_hidden1 = he_init(input_size, hidden1_size)
bias_input_hidden1 = np.zeros((1, hidden1_size))

weight_hidden1_hidden2 = he_init(hidden1_size, hidden2_size)
bias_hidden1_hidden2 = np.zeros((1, hidden2_size))

weight_hidden2_hidden3 = he_init(hidden2_size, hidden3_size)
bias_hidden2_hidden3 = np.zeros((1, hidden3_size))

weight_hidden3_hidden4 = he_init(hidden3_size, hidden4_size)
bias_hidden3_hidden4 = np.zeros((1, hidden4_size))

weight_hidden4_output = he_init(hidden4_size, output_size)
bias_hidden4_output = np.zeros((1, output_size))

In [11]:
epochs = 10000
learning_rate = 1e-2
n = x_train.shape[0]

In [12]:
for epoch in range(epochs):
    input = x_train

    hidden1_input = np.dot(input, weight_input_hidden1) + bias_input_hidden1
    hidden1_output = relu(hidden1_input)

    hidden2_input = np.dot(hidden1_output, weight_hidden1_hidden2) + bias_hidden1_hidden2
    hidden2_output = relu(hidden2_input)

    hidden3_input = np.dot(hidden2_output, weight_hidden2_hidden3) + bias_hidden2_hidden3
    hidden3_output = relu(hidden3_input)

    hidden4_input = np.dot(hidden3_output, weight_hidden3_hidden4) + bias_hidden3_hidden4
    hidden4_output = relu(hidden4_input)

    output_input = np.dot(hidden4_output, weight_hidden4_output) + bias_hidden4_output
    output = sigmoid(output_input)

    # backward pass
    error = y_train.values.reshape(-1, 1) - output
    delta_output = error * sigmoid_derivative(output)
    delta_hidden4 = delta_output.dot(weight_hidden4_output.T) * relu_derivative(hidden4_input)
    delta_hidden3 = delta_hidden4.dot(weight_hidden3_hidden4.T) * relu_derivative(hidden3_input)
    delta_hidden2 = delta_hidden3.dot(weight_hidden2_hidden3.T) * relu_derivative(hidden2_input)
    delta_hidden1 = delta_hidden2.dot(weight_hidden1_hidden2.T) * relu_derivative(hidden1_input)

    gradient_hidden4_output = hidden4_output.T.dot(delta_output) / n
    gradient_hidden3_hidden4 = hidden3_output.T.dot(delta_hidden4) / n
    gradient_hidden2_hidden3 = hidden2_output.T.dot(delta_hidden3) / n
    gradient_hidden1_hidden2 = hidden1_output.T.dot(delta_hidden2) / n
    gradient_input_hidden1 = input.T.dot(delta_hidden1) / n

    bias_hidden4_output += learning_rate * np.mean(delta_output, axis=0, keepdims=True)
    bias_hidden3_hidden4 += learning_rate * np.mean(delta_hidden4, axis=0, keepdims=True)
    bias_hidden2_hidden3 += learning_rate * np.mean(delta_hidden3, axis=0, keepdims=True)
    bias_hidden1_hidden2 += learning_rate * np.mean(delta_hidden2, axis=0, keepdims=True)
    bias_input_hidden1 += learning_rate * np.mean(delta_hidden1, axis=0, keepdims=True)

    weight_hidden4_output += learning_rate * gradient_hidden4_output
    weight_hidden3_hidden4 += learning_rate * gradient_hidden3_hidden4
    weight_hidden2_hidden3 += learning_rate * gradient_hidden2_hidden3
    weight_hidden1_hidden2 += learning_rate * gradient_hidden1_hidden2
    weight_input_hidden1 += learning_rate * gradient_input_hidden1

    if (epoch+1) % 1000 == 0 or epoch == 0:
        loss = np.mean(np.square(error))
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss}")

Epoch 1/10000 - Loss: 0.26063795953987673
Epoch 1000/10000 - Loss: 0.1509669778319373
Epoch 2000/10000 - Loss: 0.08410587376280883
Epoch 3000/10000 - Loss: 0.07575580247688304
Epoch 4000/10000 - Loss: 0.07245126320484249
Epoch 5000/10000 - Loss: 0.07052527090149224
Epoch 6000/10000 - Loss: 0.06921254073253236
Epoch 7000/10000 - Loss: 0.06827617535053042
Epoch 8000/10000 - Loss: 0.06756845679065508
Epoch 9000/10000 - Loss: 0.06700771820948116
Epoch 10000/10000 - Loss: 0.06655922809325057


In [13]:
hidden1_input = np.dot(x_test,weight_input_hidden1) + bias_input_hidden1
hidden1_output = relu(hidden1_input)

hidden2_input = np.dot(hidden1_output,weight_hidden1_hidden2) + bias_hidden1_hidden2
hidden2_output = relu(hidden2_input)

hidden3_input = np.dot(hidden2_output,weight_hidden2_hidden3) + bias_hidden2_hidden3
hidden3_output = relu(hidden3_input)

hidden4_input = np.dot(hidden3_output,weight_hidden3_hidden4) + bias_hidden3_hidden4
hidden4_output = relu(hidden4_input)

output_input = np.dot(hidden4_output,weight_hidden4_output) + bias_hidden4_output
output = sigmoid(output_input)

In [14]:
predicted_labels = (output > 0.5).astype(int)
print(classification_report(y_test, predicted_labels))

              precision    recall  f1-score   support

           0       0.85      0.92      0.88        73
           1       0.95      0.91      0.93       127

    accuracy                           0.91       200
   macro avg       0.90      0.91      0.90       200
weighted avg       0.91      0.91      0.91       200

